# Training Table Walkthrough

Builds a 7-column training table from `pre_training_table.parquet`.

| Column | Description |
|---|---|
| `t.match_id` | Match identifier |
| `t.period` | Period 1 or 2 |
| `window_time` | Window end time in seconds (0.5, 1.0, 1.5 …) |
| `data_split` | `train` / `validation` / `test` |
| `p.event_label` | Primary event in the window (`PASS`, `TACKLE`, `no event`, …) |
| `is_pass` | Binary target: 1 = PASS, 0 = anything else |
| `ball_speed_avg_xy` | Average 2-D ball speed across the window (m / frame) |

## 1. Load data

In [110]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
from IPython.display import display

from driblab.config import CONFIG_PATH, MODEL_BASE_DATA_DIR
from driblab.features import match_splits

pd.set_option('display.max_columns', None)

PRE_TRAINING_PATH = MODEL_BASE_DATA_DIR / 'pre_training_table.parquet'
OUTPUT_PATH       = MODEL_BASE_DATA_DIR / 'training_table_simple.parquet'

assert PRE_TRAINING_PATH.exists(), f'Missing {PRE_TRAINING_PATH} — run pre_training_table.ipynb first.'

df = pd.read_parquet(PRE_TRAINING_PATH)
print(f'Loaded {len(df):,} rows')
display(df[['t.match_id', 't.period', 't.frame', 't.Videotimestamp',
            'p.event_label', 'p.dist_to_actual_event',
            't.ball_x', 't.ball_y']].head(6))

Loaded 1,986,630 rows


,t.match_id,t.period,t.frame,t.Videotimestamp,p.event_label,p.dist_to_actual_event,t.ball_x,t.ball_y
0,678949,1,0,1.0,PASS,0.2,NaN,NaN
1,678949,1,1,1.1,PASS,0.1,NaN,NaN
2,678949,1,2,1.2,PASS,0.0,NaN,NaN
3,678949,1,3,1.3,PASS,0.1,NaN,NaN
4,678949,1,4,1.4,PASS,0.2,NaN,NaN
5,678949,1,5,1.5,PASS,0.3,NaN,NaN


## 2. Add data_split column

Split assignments come from `config.yaml` and are applied at the **match level** — every frame from the same match lands in the same split.
This prevents the model from seeing frames from match X in training and other frames from the same match in the test set (data leakage).

In [111]:
splits = match_splits.load_match_splits(CONFIG_PATH)

for split_name, ids in splits.items():
    print(f'  {split_name:12s}: {len(ids)} matches')

df = match_splits.add_data_split_column(df, splits, match_col='t.match_id')

print()
print('Frames per split:')
display(
    df.groupby('data_split').agg(
        frames=('t.match_id', 'count'),
        matches=('t.match_id', 'nunique'),
    ).reset_index()
)

  split_strategy: 26 matches
  train       : 23 matches
  validation  : 5 matches
  test        : 5 matches

Frames per split:


,data_split,frames,matches
0,test,292456,5
1,train,1397290,23
2,validation,296884,5


## 3. Create 5-frame non-overlapping windows

The tracking data runs at 10 Hz — one frame every 0.1 seconds.
Frames are grouped into **non-overlapping windows of exactly 5 frames** (0.5 seconds per window).

**Skip rule:**
- If a window has **fewer than 5 frames** (last partial window at the end of a period) → skip it.

Windows where ball coordinates are missing are kept. `ball_speed_avg_xy` will be `NaN` for those windows, which is valid — the model can learn from the absence of ball tracking data.

**`window_time`** counts elapsed time within each (match, period): window 1 ends at 0.5 s, window 2 at 1.0 s, and so on.
The counter resets at the start of every new (match, period).

In [112]:
# Illustrate window slicing on one (match, period)
WINDOW_SIZE = 5

sample_match  = str(df['t.match_id'].iloc[0])
sample_period = df.loc[df['t.match_id'] == sample_match, 't.period'].iloc[0]

period_df = (
    df[(df['t.match_id'] == sample_match) & (df['t.period'] == sample_period)]
    .sort_values('t.frame')
    .reset_index(drop=True)
)

n_frames    = len(period_df)
n_windows   = n_frames // WINDOW_SIZE
n_remainder = n_frames % WINDOW_SIZE

print(f'Match {sample_match}, period {sample_period}')
print(f'  Frames     : {n_frames}')
print(f'  Windows    : {n_windows}  (each = {WINDOW_SIZE} frames = 0.5 s)')
print(f'  Remainder  : {n_remainder} frame(s) discarded at end')
print()

for i in range(3):
    s = i * WINDOW_SIZE
    w = period_df.iloc[s : s + WINDOW_SIZE]
    window_time = (i + 1) * 0.5
    print(f'Window {i+1}  (window_time = {window_time} s)')
    display(
        w[['t.frame', 't.Videotimestamp', 'p.event_label',
           'p.dist_to_actual_event', 't.ball_x', 't.ball_y']]
        .reset_index(drop=True)
    )

Match 678949, period 1
  Frames     : 31658
  Windows    : 6331  (each = 5 frames = 0.5 s)
  Remainder  : 3 frame(s) discarded at end

Window 1  (window_time = 0.5 s)


,t.frame,t.Videotimestamp,p.event_label,p.dist_to_actual_event,t.ball_x,t.ball_y
0,0,1.0,PASS,0.2,NaN,NaN
1,1,1.1,PASS,0.1,NaN,NaN
2,2,1.2,PASS,0.0,NaN,NaN
3,3,1.3,PASS,0.1,NaN,NaN
4,4,1.4,PASS,0.2,NaN,NaN


Window 2  (window_time = 1.0 s)


,t.frame,t.Videotimestamp,p.event_label,p.dist_to_actual_event,t.ball_x,t.ball_y
0,5,1.5,PASS,0.3,NaN,NaN
1,6,1.6,PASS,0.4,NaN,NaN
2,7,1.7,PASS,0.5,NaN,NaN
3,8,1.8,PASS,0.6,NaN,NaN
4,9,1.9,PASS,0.7,NaN,NaN


Window 3  (window_time = 1.5 s)


,t.frame,t.Videotimestamp,p.event_label,p.dist_to_actual_event,t.ball_x,t.ball_y
0,10,2.0,PASS,0.8,NaN,NaN
1,11,2.1,PASS,0.9,NaN,NaN
2,12,2.2,no event,NaN,NaN,NaN
3,13,2.3,no event,NaN,NaN,NaN
4,14,2.4,no event,NaN,NaN,NaN


## 4. Ball speed (2D)

For each consecutive pair of frames in the window, the 2-D Euclidean distance between ball positions is computed:

$$\text{speed} = \sqrt{(x_2 - x_1)^2 + (y_2 - y_1)^2}$$

Any step where either endpoint's ball position is `NaN` is skipped.
`ball_speed_avg_xy` is the mean over all valid steps. If no valid step exists, the value is `NaN`.

The unit is **metres per frame**. At 10 Hz, multiply by 10 to get m/s.

In [113]:
# Inline example on the first window of the sample period
window = period_df.iloc[:WINDOW_SIZE].reset_index(drop=True)

print('Ball positions (5 frames):')
display(window[['t.frame', 't.ball_x', 't.ball_y']].round(3))

speeds = []
for i in range(len(window) - 1):
    x1 = window.iloc[i]['t.ball_x']
    y1 = window.iloc[i]['t.ball_y']
    x2 = window.iloc[i + 1]['t.ball_x']
    y2 = window.iloc[i + 1]['t.ball_y']

    if pd.isna(x1) or pd.isna(y1) or pd.isna(x2) or pd.isna(y2):
        print(f'  step {i}->{i+1}: skipped (NaN)')
        continue

    d = float(np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2))
    speeds.append(d)
    print(f'  step {i}->{i+1}: {d:.4f} m/frame')

ball_speed_avg_xy = float(np.mean(speeds)) if speeds else float('nan')
print(f'\nball_speed_avg_xy = {ball_speed_avg_xy:.4f} m/frame  ({ball_speed_avg_xy * 10:.2f} m/s)')

Ball positions (5 frames):


,t.frame,t.ball_x,t.ball_y
0,0,NaN,NaN
1,1,NaN,NaN
2,2,NaN,NaN
3,3,NaN,NaN
4,4,NaN,NaN


  step 0->1: skipped (NaN)
  step 1->2: skipped (NaN)
  step 2->3: skipped (NaN)
  step 3->4: skipped (NaN)

ball_speed_avg_xy = nan m/frame  (nan m/s)


## 5. Event label selection

Each frame already carries a `p.event_label` from the pre-training step.
Within a window, multiple frames can be labeled with the same event (the pre-training ±1 s window is wider than 0.5 s).

**Rule:** Among frames where `p.event_label != "no event"`, pick the one with the **smallest `p.dist_to_actual_event`**.
This selects the frame temporally closest to the moment the event occurred.
Its label becomes the window's `p.event_label`.

If the window has no labeled frames at all, `p.event_label = "no event"` and `is_pass = 0`.
                
**Tie rule (deterministic):** If two or more labeled frames in the window have exactly the same `p.dist_to_actual_event`, the code uses `idxmin()` to pick the minimum — `pandas.idxmin()` returns the first occurrence among ties. In practice this means the labeled frame with the lowest DataFrame index inside the window wins (deterministic, but dependent on row order).

**Why distance-to-actual-event?** `p.dist_to_actual_event` is computed as `|t.Videotimestamp - p.actual_event_frame|`, where `p.actual_event_frame` is the true event anchor from the events table. Choosing the frame with the smallest distance selects the frame that best represents the true event instant, rather than the frame closest to a window boundary or an arbitrary timestamp.

**Short example:** A 5-frame window contains 4 labeled frames: two `PASS` frames at distances 0.12 s and 0.20 s, and two `BALL_TOUCH` frames at 0.12 s and 0.15 s. The frames with distance 0.12 s tie across `PASS` and `BALL_TOUCH`; the tie is resolved by `idxmin()` (the earlier row in the window wins), so the window's `p.event_label` will be whichever label sits at the lower index.

If you'd prefer a different tie-breaker (e.g. prefer `PASS` over other labels, or prefer the earlier/later frame explicitly), the selection logic can be updated in `src/driblab/features/training_table.py` or the notebook — tell me which rule you want and I can implement it.

In [114]:
# Find the first window in the sample period that contains a labeled event
example_window = None
for i in range(n_windows):
    s = i * WINDOW_SIZE
    w = period_df.iloc[s : s + WINDOW_SIZE].reset_index(drop=True)
    if (w['p.event_label'] != 'no event').any():
        example_window = w
        example_idx = i
        break

if example_window is None:
    print('No labeled window found in this period.')
else:
    print(f'Window {example_idx + 1}  (window_time = {(example_idx + 1) * 0.5} s)')
    display(
        example_window[['t.frame', 't.Videotimestamp', 'p.event_label', 'p.dist_to_actual_event']]
    )

    events = example_window[example_window['p.event_label'] != 'no event']
    dist   = pd.to_numeric(events['p.dist_to_actual_event'], errors='coerce')
    closest_label = str(events.loc[dist.idxmin(), 'p.event_label'])

    print(f'\np.event_label = {closest_label}')
    print(f'is_pass       = {1 if closest_label == "PASS" else 0}')

Window 1  (window_time = 0.5 s)


,t.frame,t.Videotimestamp,p.event_label,p.dist_to_actual_event
0,0,1.0,PASS,0.2
1,1,1.1,PASS,0.1
2,2,1.2,PASS,0.0
3,3,1.3,PASS,0.1
4,4,1.4,PASS,0.2



p.event_label = PASS
is_pass       = 1


## 6. Load the training tables

The full table is built by running `src/driblab/features/training_table.py` directly.
The script writes one parquet file per split. This notebook loads all three.

In [115]:
SPLIT_PATHS = {
    split: MODEL_BASE_DATA_DIR / f'training_table_{split}.parquet'
    for split in ['train', 'validation', 'test']
}

for split_name, path in SPLIT_PATHS.items():
    assert path.exists(), (
        f'Missing {path.name}.\n'
        'Run: python src/driblab/features/training_table.py'
    )

splits = {name: pd.read_parquet(path) for name, path in SPLIT_PATHS.items()}

print('Split sizes:')
for split_name, df in splits.items():
    n_pass = int(df['is_pass'].sum())
    print(f'  {split_name:12s}: {len(df):,} windows  |  PASS: {n_pass:,} ({n_pass/len(df)*100:.1f}%)')

print()
print('Sample from train:')
display(splits['train'].head(5))

Split sizes:
  train       : 279,437 windows  |  PASS: 88,363 (31.6%)
  validation  : 59,372 windows  |  PASS: 20,073 (33.8%)
  test        : 58,488 windows  |  PASS: 21,108 (36.1%)

Sample from train:


,t.match_id,t.period,window_time,data_split,p.event_label,is_pass,ball_speed_avg_xy
0,678949,1,0.5,train,PASS,1,NaN
1,678949,1,1.0,train,PASS,1,NaN
2,678949,1,1.5,train,PASS,1,NaN
3,678949,1,2.0,train,no event,0,NaN
4,678949,1,2.5,train,no event,0,NaN


## 7. Validate

Check the output against the expected success criteria:
- Exactly 7 columns
- `ball_speed_avg_xy` may be `NaN` for windows where ball tracking was unavailable — all other columns should be complete
- `window_time` increments of 0.5
- Pass rate ~9–11 %

In [116]:
print('=== VALIDATION ===\n')

print(f'Rows    : {len(training_table):,}')
print(f'Columns : {len(training_table.columns)} (expected 7)')

missing = training_table.isnull().sum().sum()
print(f'\nMissing values: {missing}')
if missing > 0:
    print(training_table.isnull().sum()[training_table.isnull().sum() > 0])

print('\nData splits:')
for split in ['train', 'validation', 'test']:
    count = (training_table['data_split'] == split).sum()
    print(f'  {split:12s}: {count:,}')

print('\nTarget (is_pass):')
passes  = int((training_table['is_pass'] == 1).sum())
no_pass = int((training_table['is_pass'] == 0).sum())
total   = passes + no_pass
print(f'  Passes  : {passes:,} ({passes / total:.1%})')
print(f'  No-pass : {no_pass:,} ({no_pass / total:.1%})')

print('\nBall speed (m/frame):')
spd = training_table['ball_speed_avg_xy'].dropna()
print(f'  Min  : {spd.min():.3f}')
print(f'  Mean : {spd.mean():.3f}')
print(f'  Max  : {spd.max():.3f}')

print('\nWindow time (first 6 unique values):')
wt_sample = sorted(training_table['window_time'].unique())[:6]
print(f'  {wt_sample}  (expected: 0.5, 1.0, 1.5, …)')

print('\nEvent types:')
for event, count in training_table['p.event_label'].value_counts().items():
    print(f'  {event}: {count:,}')

=== VALIDATION ===

Rows    : 397,297
Columns : 7 (expected 7)

Missing values: 172051
ball_speed_avg_xy    172051
dtype: int64

Data splits:
  train       : 279,437
  validation  : 59,372
  test        : 58,488

Target (is_pass):
  Passes  : 129,544 (32.6%)
  No-pass : 267,753 (67.4%)

Ball speed (m/frame):
  Min  : 0.000
  Mean : 4.748
  Max  : 80513.743

Window time (first 6 unique values):
  [np.float64(0.5), np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0)]  (expected: 0.5, 1.0, 1.5, …)

Event types:
  no event: 230,124
  PASS: 129,544
  BALL TOUCH: 16,406
  TACKLE: 5,634
  BALL RECOVERY: 4,121
  AERIAL: 4,051
  TAKEON: 4,045
  FOUL: 3,372


## 8. Output location

The file was already written by the script. No additional save step is needed from this notebook.

In [117]:
for split_name, path in SPLIT_PATHS.items():
    print(f'{split_name:12s}: {path.relative_to(PROJECT_ROOT)}')

train       : data/processed/model_base/training_table_train.parquet
validation  : data/processed/model_base/training_table_validation.parquet
test        : data/processed/model_base/training_table_test.parquet
